# Day 2 | Notebook 5: Production RAG - The PDR Pattern
**Author: Sattaya Singkul**

Standard RAG often fails because the chunks are too small to understand (Missing Context) or too large for semantic search (Noisy Vectors). 

**The Production Strategy: Parent-Document Retrieval (PDR)**
1. **Child Chunks**: Tiny snippets (indexed for precision search).
2. **Parent Documents**: Full paragraphs (stored for context).
3. **Reranking**: An extra intelligence layer to double-check the database results.


In [5]:
!pip install redisvl sentence-transformers

## Step 1: Multi-Tier Ingestion (Child & Parent)
We store **Child Chunks** in the Vector index and the **Parent Text** in Redis JSON.


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
from redisvl.index import SearchIndex

embedder = SentenceTransformer('all-MiniLM-L6-v2')

knowledge_base = {
    "doc_1": {
        "full_text": "The Aether-X1 laptop is a beast released in 2024. It features a liquid-cooled GPU, 64GB of RAM, and is encased in Obsidian Black armor. It retails for $2,499.",
        "sub_chunks": [
            "Aether-X1 laptop liquid-cooled GPU",
            "64GB RAM high performance",
            "Obsidian Black armor casing",
            "Retails for $2,499"
        ]
    }
}

# Define Schema
schema = {
    "index": {"name": "pdr_idx", "prefix": "chunk:", "storage_type": "json"},
    "fields": [
        {"name": "parent_id", "type": "tag"},
        {"name": "chunk_text", "type": "text"},
        {"name": "embedding", "type": "vector", "attrs": {"dims": 384, "distance_metric": "cosine", "algorithm": "flat"}}
    ]
}

index = SearchIndex.from_dict(schema, redis_url="redis://redis-vector-db:6379")
index.create(overwrite=True)

# LOAD DATA
docs_to_load = []
for doc_id, data in knowledge_base.items():
    # Store Parent in standard Redis JSON (Conceptual)
    # (In this lab we use a Python dict to represent the 'Fast Fetch' storage)
    
    for chunk in data["sub_chunks"]:
        docs_to_load.append({
            "parent_id": doc_id,
            "chunk_text": chunk,
            "embedding": embedder.encode(chunk).tolist()
        })

index.load(docs_to_load)
print(f"✅ Ingested {len(docs_to_load)} precision chunks linked to parent documents.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ingested 4 precision chunks linked to parent documents.


## Step 2: Parent Retrieval & The "Hallucination Guard"
Instead of returning the tiny chunk, we find the chunk but fetch the **Parent**. 
Then, we use a **Distance Guard** to ensure we aren't talking nonsense.


In [8]:
from redisvl.query import VectorQuery

def pdr_retrieval(query, threshold=0.3):
    q_vec = embedder.encode(query).tolist()
    vq = VectorQuery(q_vec, "embedding", return_fields=["parent_id", "chunk_text"], num_results=1)
    results = index.query(vq)
    
    # 1. Cast the vector_distance string to a float for the threshold comparison
    if not results or float(results[0]['vector_distance']) > threshold:
        return None, "🚨 HALLUCINATION GUARD: No high-confidence context found."
    
    parent_id = results[0]['parent_id']
    full_context = knowledge_base[parent_id]['full_text']
    
    # 2. Cast to a float again for the math operation
    return full_context, f"Confidence: {1 - float(results[0]['vector_distance']):.2f}"

# TEST RETRIEVAL
question = "What is the GPU cooling system on the Aether?"
context, meta = pdr_retrieval(question)

print(f"Status: {meta}")
print(f"Context Found: {context}")


Status: Confidence: 0.79
Context Found: The Aether-X1 laptop is a beast released in 2024. It features a liquid-cooled GPU, 64GB of RAM, and is encased in Obsidian Black armor. It retails for $2,499.


## Step 3: Bonus - Cross-Encoder Reranking
In the highest quality systems, we do a second pass using a model that looks at the `(Question, Context)` pair together.


In [4]:
# (Conceptual realization)
# In production, we would use: 
# from sentence_transformers import CrossEncoder
# model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# scores = model.predict([question, context])

print("PRO TIP: Always use a Cross-Encoder for the final 'Top-1' ranking to ensure zero-hallucination RAG.")


PRO TIP: Always use a Cross-Encoder for the final 'Top-1' ranking to ensure zero-hallucination RAG.


### 🎉 You have mastered Production RAG Architecture!
By splitting Search and Context (PDR) and adding a Hallucination Guard, you are building robust AI systems that don't make mistakes.
